In [7]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     07_master_results.ipynb                                ║
# ║  Evaluate all trained variants and produce master comparison table         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from itertools import product
from prettytable import PrettyTable
from torch.utils.data import DataLoader

from brainvision.constants import *
from brainvision.models import (Baseline1DDNN, FabeloDNN, Fabelo2DCNN,
                                 HuEtAl1DCNN, LeeEtAl2DCNN,
                                 HamidaEtAl3DCNN, HybridSN, SpectralFormer, Simple2DCNN)
from brainvision.data import HSIPixelDataset, HSIPatchDataset
from brainvision.validation import get_splits
from brainvision.data.io import load_all_campaigns
from brainvision.utils import build_run_name
from brainvision.metrics import compute_metrics
from brainvision.history import plot_history_curves
from brainvision.device import get_device, empty_device_cache, print_device_info

# Install prettytable if needed
try:
    from prettytable import PrettyTable
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "prettytable", "-q"])
    from prettytable import PrettyTable

In [23]:
device = get_device()
print_device_info(device)

  Device     : MPS
  Name       : Apple Silicon (MPS)
  Memory     : 19,070 MB total  | N/A free  | N/A used


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
#                      CONFIGURE MASTER EVALUATION HERE
# ══════════════════════════════════════════════════════════════════════════════

MODELS = [
    '1D-NN',
    '1D-NN-Fabelo',
    '1D-CNN',
    '2D-CNN',
    '2D-CNN-Fabelo',
    '2D-CNN-Simple',
    '3D-CNN',
    'HybridSN',
    'SpectralFormer',
]

LOSS_FNS = ['CE', 'FL', 'DL', 'UFL']

STRATEGIES = [
    ('vp1',       False),   # (strategy, reduce_pixels)
    ('vp1',       True),
    ('vp2',       False),
    ('vp2',       True),
    ('vp_fabelo', False),
    ('vp_fabelo', True),
    ('lopo',      True),
]

N_FOLDS_MAP = {
    'vp1'      : 1,
    'vp2'      : 1,
    'vp3'      : 5,
    'vp_fabelo': 5,
    'lopo'     : None,   # determined from splits
}

# Output folder for training curves and results
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
# ── Registries ────────────────────────────────────────────────────────
PATCH_MODELS = {'2D-CNN', '2D-CNN-Fabelo', '2D-CNN-Simple', '3D-CNN', 'HybridSN'}

MODEL_REGISTRY = {
    '1D-NN'         : lambda: Baseline1DDNN(N_DECIMATED_BANDS, N_CLASSES,
                                             dropout=True,
                                             dropout_rate=DROPOUT_RATE),
    '1D-NN-Fabelo'  : lambda: FabeloDNN(N_DECIMATED_BANDS, N_CLASSES),
    '1D-CNN'        : lambda: HuEtAl1DCNN(N_DECIMATED_BANDS, N_CLASSES),
    '2D-CNN'        : lambda: LeeEtAl2DCNN(N_DECIMATED_BANDS, N_CLASSES,
                                             PATCH_SIZE),
    '2D-CNN-Fabelo' : lambda: Fabelo2DCNN(N_DECIMATED_BANDS, N_CLASSES,
                                         PATCH_SIZE),
    '2D-CNN-Simple' : lambda: Simple2DCNN(N_DECIMATED_BANDS, N_CLASSES,
                                           PATCH_SIZE),
    '3D-CNN'        : lambda: HamidaEtAl3DCNN(N_DECIMATED_BANDS, N_CLASSES,
                                               PATCH_SIZE),
    'HybridSN'      : lambda: HybridSN(N_DECIMATED_BANDS, PATCH_SIZE,
                                        N_CLASSES),
    'SpectralFormer': lambda: SpectralFormer(N_DECIMATED_BANDS, N_CLASSES,
                                              near_band   = SF_NEAR_BAND,
                                              dim         = SF_DIM,
                                              depth       = SF_DEPTH,
                                              heads       = SF_HEADS,
                                              dim_head    = SF_DIM_HEAD,
                                              mlp_dim     = SF_MLP_DIM,
                                              dropout     = SF_DROPOUT,
                                              emb_dropout = SF_EMB_DROPOUT,
                                              mode        = SF_MODE),
}

In [12]:
# ── Load campaigns ────────────────────────────────────────────────────

campaigns = load_all_campaigns(PROCESSED_DIRS)

Loading all campaigns from disk...
  Loaded  27 patients from ../processed/first_campaign
  Loaded  24 patients from ../processed/second_campaign
  Loaded  10 patients from ../processed/third_campaign
Loaded 61 patients across 3 campaigns



In [13]:
# ── Inference ─────────────────────────────────────────────────────────
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_preds, all_targets = [], []
    for X, y in loader:
        X      = X.to(device)
        logits = model(X)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_targets.extend(y.numpy())
    return np.array(all_preds), np.array(all_targets)


def build_test_loader(model_name: str, test_patients: list[dict]) -> DataLoader:
    if model_name in PATCH_MODELS:
        ds = HSIPatchDataset(test_patients, patch_size=PATCH_SIZE,
                                  balance=False, augment=False)
    else:
        ds = HSIPixelDataset(test_patients)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=0, pin_memory=True)

In [ ]:
# ── Master evaluation loop ───────────────────────────────────────────
# Each row: (model, loss, strategy, bal, fold, metrics_dict)
all_results   = []
skipped       = []
curves_saved  = []

total_variants = len(MODELS) * len(LOSS_FNS) * len(STRATEGIES)
checked        = 0

print(f"Scanning {total_variants} possible variants...\n")

for model_name, loss_fn, (strategy, reduce_pixels) in product(
        MODELS, LOSS_FNS, STRATEGIES):

    # Skip models not in registry
    if model_name not in MODEL_REGISTRY:
        continue

    splits = get_splits(campaigns, strategy)

    # ── Collect per-fold metrics ──────────────────────────────────────────────
    fold_metrics = []
    fold_histories = []

    for split in splits:
        run_name = build_run_name(
            model         = model_name,
            loss_fn       = loss_fn,
            strategy      = strategy,
            fold          = split['fold'],
            reduce_pixels = reduce_pixels,
        )

        ckpt_path    = Path(CHECKPOINTS_DIR) / f"{run_name}.pt"
        history_path = Path(RESULTS_DIR)     / f"{run_name}_history.npy"

        # Skip if checkpoint missing
        if not ckpt_path.exists():
            print(f"Skipping {run_name}")
            continue

        print(f"Evaluating {run_name}")

        checked += 1

        # ── Load and save training curves ────────────────────────────────────
        if history_path.exists():
            history = np.load(history_path, allow_pickle=True).item()
            fold_histories.append((run_name, history))

        # ── Run test evaluation ───────────────────────────────────────────────
        metrics_path = Path(RESULTS_DIR) / f"{run_name}_test_metrics.npy"

        if metrics_path.exists():
            # Load cached metrics if already evaluated
            m = np.load(metrics_path, allow_pickle=True).item()
            fold_metrics.append(m)
        else:
            # Run evaluation from scratch
            try:
                test_loader = build_test_loader(model_name, split['test'])
                m           = MODEL_REGISTRY[model_name]().to(device)
                m.load_state_dict(
                    torch.load(ckpt_path, map_location=device,
                               weights_only=True)
                )
                preds, targets = predict(m, test_loader)
                metrics        = compute_metrics(targets, preds)
                metrics['run_name'] = run_name
                np.save(metrics_path, metrics)
                fold_metrics.append(metrics)
                del m
                empty_device_cache()

            except Exception as e:
                print(f"  ❌ Failed: {run_name} — {e}")
                continue

    if not fold_metrics:
        skipped.append(f"{model_name}_{loss_fn}_{strategy}_"
                       f"{'bal' if reduce_pixels else 'nobal'}")
        continue

    # ── Save training curves to outputs/ ─────────────────────────────────────
    curves_dir = OUTPUT_DIR / "curves" / f"{model_name}_{loss_fn}_{strategy}"
    curves_dir.mkdir(parents=True, exist_ok=True)

    for run_name, history in fold_histories:
        out_path = plot_history_curves(history, run_name, curves_dir)
        curves_saved.append(out_path)

    # ── Aggregate across folds ────────────────────────────────────────────────
    f1_no_bg = [m['macro_f1_no_bg'] for m in fold_metrics]
    f1_all   = [m['macro_f1']       for m in fold_metrics]
    oa       = [m['oa']             for m in fold_metrics]
    dice_nbg = [m['macro_dice_no_bg'] for m in fold_metrics]
    tt_sens  = [m['sensitivity'][1] for m in fold_metrics]
    nt_sens  = [m['sensitivity'][0] for m in fold_metrics]
    bv_sens  = [m['sensitivity'][2] for m in fold_metrics]
    bg_sens  = [m['sensitivity'][3] for m in fold_metrics]
    tt_spec  = [m['specificity'][1] for m in fold_metrics]
    tt_dice  = [m['dice'][1]        for m in fold_metrics]

    n_folds = len(fold_metrics)
    bal_str = 'bal' if reduce_pixels else 'nobal'

    all_results.append({
        'model'           : model_name,
        'loss'            : loss_fn,
        'strategy'        : strategy,
        'balance'         : bal_str,
        'n_folds'         : n_folds,
        'oa'              : np.median(oa),
        'f1_all'          : np.median(f1_all),
        'f1_no_bg'        : np.median(f1_no_bg),
        'f1_no_bg_std'    : np.std(f1_no_bg),
        'dice_no_bg'      : np.median(dice_nbg),
        'nt_sens'         : np.median(nt_sens),
        'tt_sens'         : np.median(tt_sens),
        'tt_sens_std'     : np.std(tt_sens),
        'bv_sens'         : np.median(bv_sens),
        'bg_sens'         : np.median(bg_sens),
        'tt_spec'         : np.median(tt_spec),
        'tt_dice'         : np.median(tt_dice),
    })

print(f"\n✅ Evaluated {len(all_results)} variants")
print(f"⏭  Skipped  {len(skipped)} variants (no checkpoint found)")
print(f"📈 Saved {len(curves_saved)} training curve plots → {OUTPUT_DIR}/curves/")

Scanning 252 possible variants...

Evaluating 1dnn_ce_nobal_vp1
Evaluating 1dnn_ce_bal_vp1
Skipping 1dnn_ce_nobal_vp2
Skipping 1dnn_ce_bal_vp2
Evaluating 1dnn_ce_nobal_fold1_vpfabelo
Evaluating 1dnn_ce_nobal_fold2_vpfabelo
Evaluating 1dnn_ce_nobal_fold3_vpfabelo
Skipping 1dnn_ce_nobal_fold4_vpfabelo
Skipping 1dnn_ce_nobal_fold5_vpfabelo
Skipping 1dnn_ce_bal_fold1_vpfabelo
Skipping 1dnn_ce_bal_fold2_vpfabelo
Skipping 1dnn_ce_bal_fold3_vpfabelo
Skipping 1dnn_ce_bal_fold4_vpfabelo
Skipping 1dnn_ce_bal_fold5_vpfabelo
Skipping 1dnn_ce_bal_fold1_lopo
Skipping 1dnn_ce_bal_fold2_lopo
Skipping 1dnn_ce_bal_fold3_lopo
Skipping 1dnn_ce_bal_fold4_lopo
Skipping 1dnn_ce_bal_fold5_lopo
Skipping 1dnn_ce_bal_fold6_lopo
Skipping 1dnn_ce_bal_fold7_lopo
Skipping 1dnn_ce_bal_fold8_lopo
Skipping 1dnn_ce_bal_fold9_lopo
Skipping 1dnn_ce_bal_fold10_lopo
Skipping 1dnn_ce_bal_fold11_lopo
Skipping 1dnn_ce_bal_fold12_lopo
Skipping 1dnn_ce_bal_fold13_lopo
Skipping 1dnn_ce_bal_fold14_lopo
Skipping 1dnn_ce_bal_fold15_

In [17]:
# ── Master results table ─────────────────────────────────────────────
def print_master_table(results: list[dict], sort_by: str = 'f1_no_bg'):
    """Print master comparison table sorted by a given metric."""

    if not results:
        print("No results to display.")
        return

    sorted_results = sorted(results, key=lambda r: r[sort_by], reverse=True)

    table = PrettyTable()
    table.field_names = [
        "Model", "Loss", "Strategy", "Bal", "Folds",
        "OA",
        "F1 (all)",
        "F1 -BG", "F1 ±",        # ← unique names
        "Dice -BG",
        "NT Sens",
        "TT Sens", "TT Sens ±",  # ← unique names
        "BV Sens",
        "TT Spec",
        "TT Dice",
    ]

    # Alignment
    for col in ["OA", "F1 (all)", "F1 -BG", "F1 ±", "Dice -BG",
                "NT Sens", "TT Sens", "TT Sens ±", "BV Sens",
                "TT Spec", "TT Dice"]:
        table.align[col] = 'r'
    for col in ["Model", "Loss", "Strategy", "Bal", "Folds"]:
        table.align[col] = 'l'

    for r in sorted_results:
        table.add_row([
            r['model'],
            r['loss'],
            r['strategy'],
            r['balance'],
            r['n_folds'],
            f"{r['oa']*100:.1f}%",
            f"{r['f1_all']*100:.1f}%",
            f"{r['f1_no_bg']*100:.1f}%",
            f"±{r['f1_no_bg_std']*100:.1f}",
            f"{r['dice_no_bg']*100:.1f}%",
            f"{r['nt_sens']*100:.1f}%",
            f"{r['tt_sens']*100:.1f}%",
            f"±{r['tt_sens_std']*100:.1f}",
            f"{r['bv_sens']*100:.1f}%",
            f"{r['tt_spec']*100:.1f}%",
            f"{r['tt_dice']*100:.1f}%",
        ])

    print(f"\n{'═'*120}")
    print(f"  MASTER RESULTS — sorted by {sort_by}")
    print(f"  {len(sorted_results)} variants evaluated")
    print(f"  Fabelo benchmark: F1 -BG = 70.2 ± 7.9%")
    print(f"{'═'*120}")
    print(table)


# Print sorted by F1 no-BG (primary metric)
print_master_table(all_results, sort_by='f1_no_bg')

# Also print sorted by TT sensitivity — clinical priority
print_master_table(all_results, sort_by='tt_sens')


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  MASTER RESULTS — sorted by f1_no_bg
  26 variants evaluated
  Fabelo benchmark: F1 -BG = 70.2 ± 7.9%
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
+---------------+------+-----------+-------+-------+-------+----------+--------+------+----------+---------+---------+-----------+---------+---------+---------+
| Model         | Loss | Strategy  | Bal   | Folds |    OA | F1 (all) | F1 -BG | F1 ± | Dice -BG | NT Sens | TT Sens | TT Sens ± | BV Sens | TT Spec | TT Dice |
+---------------+------+-----------+-------+-------+-------+----------+--------+------+----------+---------+---------+-----------+---------+---------+---------+
| 1D-CNN        | CE   | vp2       | bal   | 1     | 96.1% |    87.7% |  84.8% | ±0.0 |    84.8% |   97.8% |   59.6% |      ±0.0 |   97.6% |   99.6% |   63.5% |
| 1D-NN-Fa

In [18]:
# ── Save master results table to CSV ─────────────────────────────────
import csv

csv_path = OUTPUT_DIR / "master_results.csv"
if all_results:
    fieldnames = all_results[0].keys()
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_results)
    print(f"✅ Master results saved → {csv_path}")

✅ Master results saved → ../outputs/master_results.csv


In [20]:
# ── Zip outputs for download ────────────────────────────────────────
import zipfile
from datetime import datetime

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
outputs_zip = Path("../") / f"outputs_{timestamp}.zip"

with zipfile.ZipFile(outputs_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUTPUT_DIR.rglob("*")):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUTPUT_DIR))

size_mb = outputs_zip.stat().st_size / 1e6
print(f"✅ Outputs zipped → {outputs_zip.name}  ({size_mb:.1f} MB)")
print(f"   Contains: {len(curves_saved)} curve plots + master_results.csv")

✅ Outputs zipped → outputs_20260724_060111.zip  (9.4 MB)
   Contains: 74 curve plots + master_results.csv
